In [22]:
import pandas as pd
import re
import numpy as np
# from deep_translator import GoogleTranslator

In [40]:
pd.set_option('display.max_rows', None)

## MAS-IT dataframes merge

In [3]:
rassereno_path = '/Users/emma/projects/cognoscolab/rassereno'
math_ma_llm = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

In [4]:
mas_it_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/mas_it_2023_humans.csv')
mas_it_humans_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/mas_it_2024_humans.csv')
mas_it_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/mas_it_2024.csv')
mas_it_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/mas_it_2025.csv')

In [5]:
mas_it_university_2025 = mas_it_2025[
    mas_it_2025['participant_id'].str.startswith('interfacce') |
    mas_it_2025['participant_id'].str.startswith('fisica')
].reset_index(drop=True)

mas_it_liceo_2025 = mas_it_2025[
    mas_it_2025['participant_id'].str.startswith('liceo')
].reset_index(drop=True)

mas_it_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/mas_it_liceo_2025.csv', index=False)

In [6]:
# get the column names of the 2025 dataset
cols_2025 = mas_it_university_2025.columns

# restrict all dataframes to those columns before concatenation
dfs = [
    mas_it_humans_2024[cols_2025.intersection(mas_it_humans_2024.columns)],
    mas_it_2023[cols_2025.intersection(mas_it_2023.columns)],
    mas_it_BA_2024[cols_2025.intersection(mas_it_BA_2024.columns)],
    mas_it_university_2025[cols_2025]
]

# concatenate them
all_universities_mas_it = pd.concat(dfs, ignore_index=True)
all_universities_mas_it.to_csv(f'{rassereno_path}/out/formatting/all_university_students/mas_it_all_universities.csv', index=False)

## Associations and Valence dataframes merge

In [7]:
associations_valence_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/associations_valence_2023_humans.csv')
associations_valence_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/associations_valence_2024_humans.csv')
associations_valence_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/associations_valence_2024.csv')
associations_valence_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/associations_valence_task_2025.csv')

In [8]:
associations_valence_university_2025 = associations_valence_2025[
    associations_valence_2025['participant_id'].str.startswith('interfacce') |
    associations_valence_2025['participant_id'].str.startswith('fisica')
]
associations_valence_liceo_2025 = associations_valence_2025[
    associations_valence_2025['participant_id'].str.startswith('liceo')
]
associations_valence_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/associations_valence_liceo_2025.csv', index=False)

In [9]:
all_universities_associations_valence = pd.concat([associations_valence_2024, associations_valence_2023, associations_valence_BA_2024, associations_valence_university_2025], ignore_index=True)
all_universities_associations_valence.to_csv(f'{rassereno_path}/out/formatting/all_university_students/associations_valence_all_universities.csv', index=False)

## Common words of multiple datasets

In [10]:
common_words_university = list(set(associations_valence_2024['cue_word']) & set(associations_valence_2023['cue_word']) & set(associations_valence_BA_2024['cue_word']) & set(associations_valence_university_2025['cue_word']))

print(f'Common word of all university subgroups: {common_words_university}')

Common word of all university subgroups: ['statistica', 'sfida', 'esame', 'matematica', 'ansia', 'fisica', 'equazione']


In [11]:
gpt_4o_associations_valence = pd.read_csv('/Users/emma/projects/cognoscolab/math_anxiety_LLMs/out/formatting/gpt_4o/associations_valence_gpt_4o.csv')

In [12]:
list(set(associations_valence_liceo_2025['cue_word']) & set(all_universities_associations_valence['cue_word']) & set(gpt_4o_associations_valence['cue_word']))

['statistica',
 'sfida',
 'matematica',
 'esame',
 'informatica',
 'innovazione',
 'ansia',
 'scienziato',
 'scienza',
 'benessere',
 'fisica',
 'equazione']

In [125]:
def cue_word_extraction(file):
    data = open(file).readlines()

    pairs = [re.findall(r'"([^"]+)"', line) for line in data]

    df = pd.DataFrame(pairs, columns=['cue_word', 'word'])

    counts = df['cue_word'].value_counts()

    cue_words = counts[:len(counts) // 2]

    cue_words_list = cue_words.index.tolist()

    return df, cue_words_list

In [14]:
experts_df, cue_words_experts = cue_word_extraction('/Users/emma/Desktop/data_2021/Experts_unfiltered.txt')

In [126]:
students_df_unfiltered, cue_words_students_unfiltered = cue_word_extraction('/Users/emma/Desktop/data_2021/Students_unFiltered.txt')

In [127]:
students_df_filtered, cue_words_students_filtered = cue_word_extraction('/Users/emma/Desktop/data_2021/Students_Filtered.txt')

In [107]:
list(set(associations_valence_liceo_2025['cue_word']) & set(all_universities_associations_valence['cue_word']) & set(gpt_4o_associations_valence['cue_word']) & set(cue_words_students_unfiltered))

['esame', 'fisica', 'scienziato', 'matematica']

## Check if valences are for filtered or unfiltered df

In [ ]:
# counts how many valence ratings per word exist in ValenceStudentsUPD.txt
counts = {}
with open("/Users/emma/Desktop/data_2021/ValenceStudentsUPD.txt", "r") as f:
    for line in f:
        parts = line.strip().split()
        # gets the first element of each row, which is the word
        word = parts[0]
        # gets all the numbers after the word, removing the .
        numbers = [p for p in parts[1:] if p.replace('.', '').isdigit()]
        # counts how many valence ratings per word exist
        counts[word] = counts.get(word, 0) + len(numbers)
df_counts = pd.DataFrame(list(counts.items()), columns=['word', 'count_valence'], index=None)

In [103]:
def count_students_valence(df, file_name):
    combined_counts = (
        df['cue_word'].value_counts()
        .add(df['word'].value_counts(), fill_value=0)
        .astype(int)
    )

    combined_counts = pd.DataFrame(list(combined_counts.items()), columns=['word', 'count_cue_ass_word'], index=None)

    df = pd.merge(df_counts, combined_counts, on='word', how='outer')
    df.to_csv(f'/Users/emma/projects/cognoscolab/rassereno/out/formatting/students_2021/combined_counts_{file_name}.csv', index=False)

In [104]:
count_students_valence(students_df_unfiltered, 'unfiltered')
count_students_valence(students_df_filtered, 'filtered')

## Creation df for word and associations

In [100]:
valences = open("/Users/emma/Desktop/data_2021/ValenceStudentsUPD.txt", "r").readlines()
valences = [v.strip().split('\t') for v in valences]

In [152]:
valence_map = {}
for row in valences:
    if not row[0].replace('.', '').isdigit():
        word = row[0]
        vals = [v for v in row[1:] if v.replace('.', '').isdigit()]
        valence_map.setdefault(word, []).extend(vals)

# Assign valences sequentially
def assign_valence(row, col_name, valence_map):
    word = row[col_name]
    if word in valence_map and len(valence_map[word]) > 0:
        return float(valence_map[word].pop(0).replace('.', ''))
    else:
        return np.nan

students_df_unfiltered['valence_association_1'] = students_df_unfiltered.apply(assign_valence, axis=1, args=(('word', valence_map)))
students_df_unfiltered['valence_association_0'] = students_df_unfiltered.apply(assign_valence, axis=1, args=(('cue_word', valence_map)))

col_order = ['cue_word', 'valence_association_0', 'word', 'valence_association_1']
students_df_unfiltered = students_df_unfiltered[col_order]

In [153]:
# students_filtered = students_df_unfiltered.merge(
#     students_df_filtered[['cue_word', 'word']],
#     on=['cue_word', 'word'],
#     how='inner'
# )

students_df_unfiltered['new_index'] = students_df_unfiltered.index + 1
students_df_unfiltered['cue_word_original'] = students_df_unfiltered['cue_word']

In [163]:
df = students_df_unfiltered.rename(columns={
    'cue_word': 'association_0',
    'valence_association_0': 'valence_association_0',
    'word': 'association_1',
    'valence_association_1': 'valence_association_1'
})

# melt from wide to long
df_long = pd.wide_to_long(
    df,
    stubnames=['association', 'valence_association'],
    i=['new_index', 'cue_word_original'],  # keeps your original row index as unique ID
    j='index',
    sep='_',
    suffix=r'\d+'
).reset_index()

# optional: rename columns for clarity
df_long = df_long.rename(columns={
    'association': 'word',
    'valence_association': 'valence'
})

df_long.rename(columns={
    'cue_word_original': 'cue_word'}, inplace=True)
df_long

,new_index,cue_word,index,word,valence
0,1,esame,0,esame,2.0
1,1,esame,1,test,5.0
2,2,esame,0,esame,1.0
3,2,esame,1,test,3.0
4,3,test,0,test,5.0
...,...,...,...,...,...
23451,11726,piatto,1,forchetta,4.0
23452,11727,produzione,0,produzione,4.0
23453,11727,produzione,1,audiometria,5.0
23454,11728,produzione,0,produzione,NaN


In [ ]:
columns_to_clean = ['cue_word', 'word']
for col in columns_to_clean:
    if col in df_long.columns:
        df_long[col] = (
            df_long[col]
            .astype(str)
            .str.lower()
            .str.replace(r'[\d\W_]+', ' ', regex=True)
            .str.strip()
        )
df_long = df_long.dropna()

df_long = df_long[(df_long['word'] != 'nan') & (df_long['word'] != '')]
df_long

,new_index,cue_word,index,word,valence
0,1,esame,0,esame,2.0
1,1,esame,1,test,5.0
2,2,esame,0,esame,1.0
3,2,esame,1,test,3.0
4,3,test,0,test,5.0
...,...,...,...,...,...
23449,11725,piatto,1,cucchiaio,4.0
23451,11726,piatto,1,forchetta,4.0
23452,11727,produzione,0,produzione,4.0
23453,11727,produzione,1,audiometria,5.0


In [165]:
df_long.to_csv('/Users/emma/projects/cognoscolab/rassereno/out/formatting/students_2021/associations_valence_students_2021.csv', index=False)